# 01 - Qualité des données

**Objectif** : mesurer ce que vaut la table de référence `data/processed/products.parquet` avant toute analyse.

On répond à quatre questions :
1. Combien de produits perd-on à chaque étape du nettoyage, et pourquoi ?
2. Combien de valeurs aberrantes ont été neutralisées ?
3. Quelles colonnes sont fiables, lesquelles sont lacunaires ?
4. Sur combien de produits repose chaque objectif du projet ?

Prérequis : avoir lancé `uv run python -m nutrilys.data.clean`.

In [1]:
import json

import pandas as pd
import plotly.express as px

from nutrilys import config
from nutrilys.data.schema import PRODUCTS_SCHEMA
from nutrilys.db import connect, sql_path

pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.1f}".replace(",", " "))

con = connect()
con.execute(
    f"CREATE VIEW products AS SELECT * FROM read_parquet('{sql_path(config.PRODUCTS_PATH)}')"
)
report = json.loads(config.CLEANING_REPORT_PATH.read_text(encoding="utf-8"))

n_total = con.execute("SELECT count(*) FROM products").fetchone()[0]
print(f"Table de référence : {n_total:,} produits".replace(",", " "))
print(f"Générée le {report['generated_at']} à partir de {report['source']}")

Table de référence : 1 171 729 produits
Générée le 2026-11-12T22:42:52 à partir de off_products.parquet


## 1. Entonnoir de nettoyage

Un produit n'est retiré que s'il est en double (même code-barres, on garde la fiche la plus récente) ou s'il n'a pas de nom. Toutes les autres anomalies sont traitées valeur par valeur, sans supprimer le produit.

In [2]:
steps = pd.DataFrame(report["steps"]).rename(columns={"step": "Étape", "rows": "Produits"})
steps["Retirés"] = (-steps["Produits"].diff()).fillna(0).astype(int)
steps["Conservés (%)"] = 100 * steps["Produits"] / steps["Produits"].iloc[0]
steps

,Étape,Produits,Retirés,Conservés (%)
0,Extrait France,1296967,0,100.0
1,Un produit par code-barres,1296939,28,100.0
2,Avec un nom de produit,1171729,125210,90.3
3,Table de référence,1171729,0,90.3


In [3]:
kept = steps["Conservés (%)"].iloc[-1]
fig = px.funnel(
    steps,
    x="Produits",
    y="Étape",
    title=f"{kept:.1f} % des produits extraits sont conservés dans la table de référence",
)
fig.update_layout(template="simple_white", height=380)
fig.show()

## 2. Valeurs neutralisées

Deux types de corrections, appliquées valeur par valeur :
- **Scores hors domaine** : Open Food Facts stocke « unknown » ou « not-applicable » dans le champ Nutri-Score quand le score n'est pas calculable. Ces valeurs sont mises à vide.
- **Nutriments aberrants** : valeur négative, supérieure à 100 g pour 100 g (900 kcal pour l'énergie), ou sous-total supérieur au total (graisses saturées supérieures aux graisses, sucres supérieurs aux glucides, avec 1 g de tolérance d'arrondi).

In [4]:
scores = pd.Series(report["invalid_scores_set_to_null"], name="Valeurs mises à vide")
nutrients = pd.Series(report["nutrient_values_set_to_null"], name="Valeurs mises à vide")
corrections = pd.concat([scores, nutrients]).to_frame()
corrections["Part de la table (%)"] = 100 * corrections["Valeurs mises à vide"] / n_total
corrections

,Valeurs mises à vide,Part de la table (%)
nutriscore_grade,697575,59.5
nova_group,0,0.0
energy_kcal_100g,784,0.1
fat_100g,37,0.0
saturated_fat_100g,182,0.0
carbohydrates_100g,79,0.0
sugars_100g,390,0.0
fiber_100g,39,0.0
proteins_100g,26,0.0
salt_100g,42,0.0


## 3. Taux de remplissage

Pour les colonnes de type liste (catégories, labels, additifs), une liste vide compte comme non renseignée.

In [5]:
def filled(column: str, kind: str) -> str:
    """Nombre de valeurs renseignées ; une liste vide compte comme absente."""
    if kind == "liste":
        return f"count(*) FILTER (WHERE len({column}) > 0)"
    return f"count({column})"


fill_exprs = [
    f"100.0 * {filled(column, kind)} / count(*) AS {column}"
    for column, (kind, _) in PRODUCTS_SCHEMA.items()
    if kind != "booléen" and column != "code"
]

fill = (
    con.execute(f"SELECT {', '.join(fill_exprs)} FROM products")
    .df()
    .T.rename(columns={0: "Rempli (%)"})
    .sort_values("Rempli (%)")
)
fill["Type"] = [PRODUCTS_SCHEMA[c][0] for c in fill.index]

In [6]:
fig = px.bar(
    fill.reset_index(names="Colonne"),
    x="Rempli (%)",
    y="Colonne",
    orientation="h",
    title="Les scores et nutriments sont bien renseignés, les ingrédients et NOVA beaucoup moins",
)
fig.update_layout(template="simple_white", height=750, xaxis_range=[0, 100])
fig.show()

Le titre du graphique ci-dessus est à vérifier sur les données réelles : il reprend le constat de l'extraction (étape 1), où ingrédients et NOVA étaient renseignés pour moins d'un produit sur trois.

## 4. Sous-populations par objectif

Chaque objectif ne peut s'appuyer que sur les produits qui ont les informations nécessaires. Ce tableau fixe les effectifs de référence pour la suite du projet.

In [7]:
subsets = (
    con.execute("""
    SELECT
        count(*) AS "Tous les produits (recherche dans l'app)",
        count(*) FILTER (has_nutriscore) AS "Nutri-Score valide (objectif 1)",
        count(*) FILTER (has_nutrients) AS "Sucre, sel et graisses saturées (objectif 1)",
        count(*) FILTER (has_nova AND has_ingredients) AS "NOVA et ingrédients (objectif 3)",
        count(*) FILTER (has_nutriscore AND has_nova AND has_ingredients)
            AS "Nutri-Score, NOVA et ingrédients (objectif 2)",
        count(*) FILTER (is_organic) AS "Produits bio déclarés",
        count(*) FILTER (is_organic AND has_nutriscore AND food_group IS NOT NULL)
            AS "Bio avec Nutri-Score et groupe alimentaire (objectif 5)"
    FROM products
""")
    .df()
    .T.rename(columns={0: "Produits"})
)
subsets["Part (%)"] = 100 * subsets["Produits"] / n_total
subsets

,Produits,Part (%)
Tous les produits (recherche dans l'app),1171729,100.0
Nutri-Score valide (objectif 1),471331,40.2
"Sucre, sel et graisses saturées (objectif 1)",829765,70.8
NOVA et ingrédients (objectif 3),319524,27.3
"Nutri-Score, NOVA et ingrédients (objectif 2)",257126,21.9
Produits bio déclarés,138829,11.8
Bio avec Nutri-Score et groupe alimentaire (objectif 5),62915,5.4


In [8]:
fig = px.bar(
    subsets.reset_index(names="Sous-population").iloc[::-1],
    x="Produits",
    y="Sous-population",
    orientation="h",
    text="Produits",
    title="Effectifs disponibles pour chaque objectif",
)
fig.update_traces(texttemplate="%{text:,.0f}")
fig.update_layout(template="simple_white", height=420, separators=", ")
fig.show()

## 5. Contrôle des scores

Distributions brutes, pour vérifier qu'il ne reste que des valeurs valides. L'analyse proprement dite est l'objet de l'étape 3.

In [9]:
nutriscore = con.execute("""
    SELECT coalesce(nutriscore_grade, 'vide') AS "Nutri-Score", count(*) AS "Produits"
    FROM products GROUP BY 1 ORDER BY 1
""").df()
nova = con.execute("""
    SELECT coalesce(nova_group::VARCHAR, 'vide') AS "NOVA", count(*) AS "Produits"
    FROM products GROUP BY 1 ORDER BY 1
""").df()
display(nutriscore, nova)

,Nutri-Score,Produits
0,a,64301
1,b,50320
2,c,98291
3,d,124462
4,e,133957
5,vide,700398


,NOVA,Produits
0,1,45113
1,2,27260
2,3,73100
3,4,195133
4,vide,831123


## 6. Couverture des groupes alimentaires

Le groupe alimentaire (nomenclature PNNS utilisée par Open Food Facts) servira à comparer à catégorie égale, en particulier pour le bio. Un groupe trop petit ne permettra pas de conclure.

In [10]:
groups = con.execute("""
    SELECT
        coalesce(food_group, 'non renseigné') AS "Groupe alimentaire",
        count(*) AS "Produits",
        100.0 * avg(has_nutriscore::INT) AS "Avec Nutri-Score (%)",
        100.0 * avg(has_nova::INT) AS "Avec NOVA (%)",
        100.0 * avg(is_organic::INT) AS "Bio (%)"
    FROM products
    GROUP BY 1
    ORDER BY 2 DESC
""").df()
groups

,Groupe alimentaire,Produits,Avec Nutri-Score (%),Avec NOVA (%),Bio (%)
0,non renseigné,615988,5.9,8.2,9.5
1,sugary-snacks,117779,80.5,59.6,13.5
2,fish-meat-eggs,98329,80.8,35.5,6.6
3,milk-and-dairy-products,63210,86.2,48.1,11.2
4,cereals-and-potatoes,58093,83.4,55.3,26.6
5,beverages,47436,70.3,64.0,21.5
6,fruits-and-vegetables,41874,74.1,48.8,21.1
7,fats-and-sauces,39613,79.5,67.8,16.8
8,composite-foods,39137,90.2,56.4,8.0
9,salty-snacks,30150,88.0,49.9,11.9


## 7. Dictionnaire de données

Également exporté dans `docs/data_dictionary.md`.

In [11]:
pd.DataFrame(
    [(name, kind, desc) for name, (kind, desc) in PRODUCTS_SCHEMA.items()],
    columns=["Colonne", "Type", "Description"],
)

,Colonne,Type,Description
0,code,texte,"Code-barres du produit, identifiant unique."
1,product_name,texte,"Nom du produit, langue principale ou à défaut ..."
2,generic_name,texte,Dénomination générique (« biscuits fourrés au ...
3,ingredients_text,texte,Liste des ingrédients telle que saisie sur Ope...
4,brand,texte,"Première marque déclarée, pour l'affichage."
5,brand_tag,texte,Première marque normalisée par Open Food Facts...
6,categories_tags,liste,"Catégories Open Food Facts, de la plus large à..."
7,main_category,texte,Catégorie de référence utilisée par Open Food ...
8,food_group,texte,"Groupe alimentaire de niveau 1 (PNNS), ex. sug..."
9,food_subgroup,texte,"Sous-groupe alimentaire de niveau 2 (PNNS), ex..."


## 8. Limites

- **Données collaboratives** : Open Food Facts est rempli par des bénévoles et des producteurs. Une information absente ne veut pas dire que la réalité est absente. Un label bio non déclaré ne signifie pas qu'un produit n'est pas bio.
- **Biais de sélection** : les produits avec liste d'ingrédients (et donc NOVA) sont probablement plus populaires ou plus récents que la moyenne. Les résultats sur NOVA décrivent cette sous-population, pas tout le marché.
- **Instantané** : la base évolue chaque jour. Les chiffres valent pour l'export du `generated_at` affiché en haut du notebook.
- **Nutri-Score** : il s'agit de la version 2023 de l'algorithme, calculée par Open Food Facts. Elle peut différer du logo imprimé sur l'emballage.